In [ ]:
import torch
from psi_net.cartesian import CartesianSchrodinger, CartesianCoordinates, CartesianInitialCondition
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
a = 1.0
V = lambda t, x: 0.0
t = torch.linspace(0, 1, 100)
x = torch.linspace(0, a, 100)
hbar = 1.0
m = 1.0
N = 2 # Number of energy levels to include in the initial wavefunction

def psi_n(x, n):
    try:
        return np.sqrt(2/a)*torch.sin(n* np.pi * x / a)
    except Exception as e:
        return np.sqrt(2/a)*np.sin(n* np.pi * x / a)

def initial_wavefunction(t, x):
    return 1/np.sqrt(N)*sum(psi_n(x, n) for n in range(1, N+1))

def psi_r_theoretical(t, x):
    return sum(psi_n(x, n) * np.cos(n**2 * np.pi**2 * hbar * t / (2 * m * a**2)) for n in range(1, N+1))

def psi_i_theoretical(t, x):
    return sum(-psi_n(x, n) * np.sin(n**2 * np.pi**2 * hbar * t / (2 * m * a**2)) for n in range(1, N+1))

def zero_initial(t, x):
    return torch.zeros_like(x)

def zero_at_boundaries(t, x):
    return torch.zeros_like(t)

initial_conditions = [
    CartesianInitialCondition(CartesianCoordinates.T, 0.0, initial_wavefunction, zero_initial),
    CartesianInitialCondition(CartesianCoordinates.X, 0.0, zero_at_boundaries, zero_at_boundaries),
    CartesianInitialCondition(CartesianCoordinates.X, a, zero_at_boundaries, zero_at_boundaries)
]

schrodinger = CartesianSchrodinger(
    [t,x], 
    V, 
    initial_conditions, 
    hbar, 
    m,
    hidden_size=128,
    num_layers=4
)

In [ ]:
schrodinger.train(
    num_epochs=500, 
    de_weight=1.0, 
    ic_weight=0.5, 
    print_every=50
)
# schrodinger.train(
#     num_epochs=200, 
#     de_weight=0.1, 
#     ic_weight=1.0, 
#     print_every=10
# )
# schrodinger.train(
#     num_epochs=100,
#     de_weight=1.0,
#     ic_weight=0.5,
#     print_every=10
# )
# num_phases = 4
# for phase in range(1, num_phases+1):
#     alpha = phase / num_phases
#     schrodinger.train(
#         num_epochs=100, 
#         de_weight=1.0*alpha, 
#         ic_weight=1.0 * (1-alpha), 
#         print_every=10
#     )

In [ ]:
solution_df = schrodinger.get_solution()
psi_R = solution_df['psi_R'].values
psi_I = solution_df['psi_I'].values
X = solution_df['x'].values
T = solution_df['t'].values
prob = solution_df['probability_density'].values
solution_df.head()


In [ ]:
psi_R_0 = solution_df[solution_df['t'] == 0]['psi_R'].values
psi_I_0 = solution_df[solution_df['t'] == 0]['psi_I'].values
X_0 = solution_df[solution_df['t'] == 0]['x'].values
plt.scatter(X_0, initial_wavefunction(0, X_0), label='Initial Wavefunction')
plt.scatter(X_0, psi_R_0, label='psi_R (NN)')
plt.scatter(X_0, psi_I_0, label='psi_I (NN)')
plt.xlabel('Position')
plt.ylabel('Wavefunction')
plt.legend()
plt.show()

In [ ]:
psi_r_t = psi_r_theoretical(T, X)
psi_i_t = psi_i_theoretical(T, X)
prob_theoretical = psi_r_t**2 + psi_i_t**2

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

# ax.scatter(
#     T, 
#     X,
#     psi_r_t, 
#     c=psi_r_t, 
#     cmap='magma',
#     label='Theoretical Real Part',
#     alpha=0.5
# )

# ax.scatter(
#     T, 
#     X,
#     psi_R, 
#     c=psi_R, 
#     cmap='viridis',
#     label='Neural Network Real Part',
#     alpha=0.5
# )
ax.scatter(T, X, psi_R-psi_r_t, c=psi_R-psi_r_t, cmap='coolwarm', label='NN - Theoretical', alpha=0.5)
ax.set_xlabel('Time')
ax.set_ylabel('Position')
ax.set_zlabel('Real Part of Wavefunction')
plt.legend()
plt.title("Real Part")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

# ax.scatter(
#     T, 
#     X,
#     psi_i_t, 
#     c=psi_i_t, 
#     cmap='magma',
#     label='Theoretical Imaginary Part',
#     alpha=0.5
# )
# ax.scatter(
#     T, 
#     X,
#     psi_I, 
#     c=psi_I, 
#     cmap='viridis',
#     label='Neural Network Imaginary Part',
#     alpha=0.5
# )
ax.scatter(T, X, psi_I-psi_i_t, c=psi_I-psi_i_t, cmap='coolwarm', label='NN - Theoretical', alpha=0.5)
ax.set_xlabel('Time')
ax.set_ylabel('Position')
ax.set_zlabel('Imaginary Part of Wavefunction')
plt.title("Imaginary Part")
plt.legend()
plt.show()

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

# ax.scatter(
#     T, 
#     X,
#     prob_theoretical, 
#     c=prob_theoretical, 
#     cmap='magma',
#     label='Theoretical Probability Density',
#     alpha=0.5
# )
# ax.scatter(
#     T, 
#     X,
#     prob, 
#     c=prob, 
#     cmap='viridis',
#     label='Neural Network Probability Density',
#     alpha=0.5
# )
ax.scatter(T, X, prob-prob_theoretical, c=prob-prob_theoretical, cmap='coolwarm', label='Difference between Theoretical and Neural Network', alpha=0.5)
ax.set_xlabel('Time')
ax.set_ylabel('Position')
ax.set_zlabel('Probability Density')
plt.legend()
plt.title("Probability Density")
plt.show()

In [ ]:
schrodinger.de_loss(schrodinger.inputs)

In [ ]:
p_i = schrodinger.model(schrodinger.inputs)[:,1]
p_r = schrodinger.model(schrodinger.inputs)[:,0]

In [ ]:
real_part = schrodinger._dt(p_i, schrodinger.inputs) - schrodinger._laplacian(p_r, schrodinger.inputs) + schrodinger._V(schrodinger.inputs) * p_r
imaginary_part = -schrodinger._dt(p_r, schrodinger.inputs) - schrodinger._laplacian(p_i, schrodinger.inputs) + schrodinger._V(schrodinger.inputs) * p_i

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

# ax.scatter(T, X, real_part.detach().numpy(), c=real_part.detach().numpy(), cmap='coolwarm', label='Real Part of DE Loss', alpha=0.5)
ax.scatter(T, X, imaginary_part.detach().numpy(), c=imaginary_part.detach().numpy(), cmap='viridis', label='Imaginary Part of DE Loss', alpha=0.5)

In [ ]:
dI_dt = schrodinger._dt(p_i, schrodinger.inputs)
dR_dt = schrodinger._dt(p_r, schrodinger.inputs)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
# ax.scatter(T, X, dI_dt.detach().numpy(), c=dI_dt.detach().numpy(), cmap='magma', label='dI/dt', alpha=0.5)
ax.scatter(T, X, dR_dt.detach().numpy(), c=dR_dt.detach().numpy(), cmap='magma', label='dR/dt', alpha=0.5)
plt.show()

In [ ]:
l_r = schrodinger._laplacian(p_r, schrodinger.inputs)
l_i = schrodinger._laplacian(p_i, schrodinger.inputs)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(T, X, l_r.detach().numpy(), c=l_r.detach().numpy(), cmap='magma', label='Laplacian R', alpha=0.5)
ax.scatter(T, X, l_i.detach().numpy(), c=l_i.detach().numpy(), cmap='magma', label='Laplacian I', alpha=0.5)
plt.show()

In [ ]:
from math_utils import diff

def _laplacian(psi, X):
        laplacian = torch.zeros_like(psi)
        for i in range(1, X.shape[1]):
            laplacian += diff(psi, X, coordinate=i, order=2)
        return laplacian

diff(p_r, schrodinger.inputs, coordinate=1, order=2)